In [2]:
import os
import random
import shutil

# Define paths for source and destination directories
src_dir = 'D:\Github\Skincancer-Prediction\Model Training\Dataset'
train_dir = 'D:\Github\Skincancer-Prediction\Model Training\\New Dataset\\train'
test_dir = 'D:\Github\Skincancer-Prediction\Model Training\\New Dataset\\test'

# Define the fraction of images to be used for testing
main_fraction = 0.5
test_fraction = 0.2

# Loop over each subdirectory in the source directory
for subdir in os.listdir(src_dir):
    subdir_path = os.path.join(src_dir, subdir)
    if os.path.isdir(subdir_path):
        print(f"Processing {subdir}...")
        img_files = os.listdir(subdir_path)
        num_test = int(len(img_files))
        
        # Shuffle the image files randomly
        random.shuffle(img_files)
        
        # Copy the first 'num_test' images to the testing directory
        for img_file in img_files[:int(num_test*0.2)]:
            src_path = os.path.join(subdir_path, img_file)
            dst_path = os.path.join(test_dir, subdir, img_file)
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            shutil.copy(src_path, dst_path)
        
        # Copy the remaining images to the training directory
        for img_file in img_files[int(num_test*0.2):]:
            src_path = os.path.join(subdir_path, img_file)
            dst_path = os.path.join(train_dir, subdir, img_file)
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            shutil.copy(src_path, dst_path)

print("Done.")

Processing melanoma...
Processing others...
Done.


In [1]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  0


In [3]:
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
config = ConfigProto()
config.gpu_options.allow_growth = True
session = InteractiveSession(config=config)
import numpy as np
import pandas as pd
import os
from sklearn.datasets import load_files
from keras.utils import np_utils 

c:\Users\bmpra\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\python\client\session.py:1769: UserWarning: An interactive session is already active. This can cause out-of-memory errors in some cases. You must explicitly call `InteractiveSession.close()` to release resources held by the other session(s).
  warnings.warn('An interactive session is already active. This can '


In [4]:
path = "./Dataset/"
data = load_files(path)

In [5]:
print("Filename: \n", data['filenames'][:405])
print("Targets: \n", data['target'][:405])

Filename: 
 ['./Dataset/others\\ISIC_0029684.jpg'
 './Dataset/melanoma\\ISIC_0032408.jpg'
 './Dataset/melanoma\\ISIC_0027063_90_angle.jpg'
 './Dataset/melanoma\\ISIC_0033902_270_angle_flipped.jpg'
 './Dataset/others\\ISIC_0029883.jpg'
 './Dataset/melanoma\\ISIC_0034068_180_angle.jpg'
 './Dataset/melanoma\\ISIC_0031005_270_angle.jpg'
 './Dataset/others\\ISIC_0027134.jpg'
 './Dataset/melanoma\\ISIC_0027204_270_angle.jpg'
 './Dataset/others\\ISIC_0024448.jpg' './Dataset/others\\ISIC_0027599.jpg'
 './Dataset/melanoma\\ISIC_0032596_270_angle_flipped.jpg'
 './Dataset/others\\ISIC_0033771.jpg' './Dataset/others\\ISIC_0030021.jpg'
 './Dataset/melanoma\\ISIC_0034216_270_angle.jpg'
 './Dataset/others\\ISIC_0033969.jpg'
 './Dataset/melanoma\\ISIC_0029033_flipped.jpg'
 './Dataset/melanoma\\ISIC_0033180.jpg'
 './Dataset/melanoma\\ISIC_0028325_90_angle.jpg'
 './Dataset/melanoma\\ISIC_0029913_180_angle.jpg'
 './Dataset/melanoma\\ISIC_0025153.jpg'
 './Dataset/melanoma\\ISIC_0027261_270_angle_flipped.j

In [6]:
data['target']

array([1, 0, 0, ..., 1, 1, 0])

In [7]:
from tensorflow.keras.utils import to_categorical
target = to_categorical(np.array(data['target']), num_classes=2)
target

array([[0., 1.],
       [1., 0.],
       [1., 0.],
       ...,
       [0., 1.],
       [0., 1.],
       [1., 0.]], dtype=float32)

In [8]:
len(data['filenames'])

16707

In [9]:
# Splitting the data into the training and validation set
train_files, train_targets = data['filenames'][:300], data['target'][:300]
valid_files, valid_targets = data['filenames'][300:], data['target'][300:]

In [10]:
# Importing the libraries
import keras
from keras.preprocessing import image          
from tqdm import tqdm
from PIL import ImageFile                            
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [11]:
def path_to_tensor(img_path):
    """
    Getting a tensor from a given path.
    """
    # Loading the image
    img = image.load_img(img_path, target_size=(512, 512))
    # Converting the image to numpy array
    x = image.img_to_array(img)   
    # convert 3D tensor to 4D tensor with shape (1, 512, 512, 3)
    return np.expand_dims(x, axis=0)

def paths_to_tensor(img_paths):
    """
    # Getting a list of tensors from a given path directory.
    """
    list_of_tensors = [path_to_tensor(img_path) for img_path in tqdm(img_paths)]
    return np.vstack(list_of_tensors)

In [ ]:
# pre-process the data for Keras
train_tensors = paths_to_tensor(train_files).astype('float16')/255
valid_tensors = paths_to_tensor(valid_files).astype('float16')/255

In [ ]:
# Saving the data
np.save("./Saved image tensors/augmented_training_tensors.npy", train_tensors)
np.save("./Saved image tensors/augmented_validation_tensors.npy", valid_tensors)

In [ ]:
# Loading the data
train_tensors = np.load("./Saved image tensors/augmented_training_tensors.npy")
valid_tensors = np.load("./Saved image tensors/augmented_validation_tensors.npy")

In [13]:
from tensorflow.keras.applications.mobilenet import MobileNet
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint

In [ ]:
def mobilenet_architecture():
    base_model = MobileNet(include_top=False, weights=None, input_shape=(512, 512, 3))
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    output = Dense(units=2, activation='softmax')(x)
    mobilenet_model = Model(base_model.input, output)
    mobilenet_model.compile(optimizer=Adam(learning_rate=0.001),
                            loss='categorical_crossentropy',
                            metrics=['accuracy'])
    return mobilenet_model

In [ ]:
# Getting the mobilenet
mobilenet_model = mobilenet_architecture()  

In [ ]:
checkpointer = ModelCheckpoint(filepath='Saved models/weights.best.mobilenet.hdf5', 
                               verbose=1, 
                               save_best_only=True)

mobilenet_model.fit(train_tensors, 
                    train_targets, 
                    batch_size = 1,
                    validation_data = (valid_tensors, valid_targets),
                    epochs = 5,
                    callbacks=[checkpointer], 
                    verbose=1)

In [ ]:
# Loading the weights
mobilenet_model.load_weights("./Saved models/weights.best.mobilenet.hdf5")

Bio Inspired Optimisation(PSO)

In [ ]:
model_architecture = mobilenet_architecture()
weight_path = "./Saved models bio/weights.best.mobilenet.hdf5"

In [ ]:
def predict(img_path, 
            model_architecture = model_architecture, 
            path_model_weight = weight_path):
    # Getting the tensor of image
    image_to_predict = path_to_tensor(img_path).astype('float16')/255
    # Getting the model's architecture
    model = model_architecture
    # Loading the weights
    model.load_weights(path_model_weight)
    # Predicting
    pred = model.predict(image_to_predict)
    print("Prediction..." + " Melanoma : ", pred[0][0], " | Other : ", pred[0][1])
    if np.argmax(pred) == 0:
        return [1., 0.]
    elif np.argmax(pred) == 1:
        return [0., 1.]

In [ ]:
# Import required libraries
import numpy as np
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from pyswarm import pso

# Define the fitness function to optimize
def fitness(params):
    # Extract the learning rate and batch size parameters
    # lr, batch_size, beta_1, beta_2, epsilon = params
    lr = params[0]
    print(lr,params,
        #    batch_size, beta_1, beta_2, epsilon,
           "------------------------")
    
    # Define the model
    model = mobilenet_architecture()
    
    # Compile the model with the given learning rate
    model.optimizer = Adam(learning_rate=lr)
    
    # Train the model with the given batch size
    early_stopping = EarlyStopping(patience=5, restore_best_weights=True)
    checkpointer1 = ModelCheckpoint(filepath='Saved models bio/weights.best.mobilenet.hdf5', 
                               verbose=1, 
                               save_best_only=True)
    model.fit(train_tensors, train_targets,batch_size=4, epochs=5, validation_split= (valid_tensors, valid_targets), callbacks=[early_stopping, checkpointer1], verbose=0)
    
    # # Evaluate the model on the testing set
    # y_pred = model.predict(X_test)
    # y_pred = np.argmax(y_pred, axis=1)
    # acc = accuracy_score(np.argmax(y_test, axis=1), y_pred)

    # Compute test set predictions
    NUMBER_TEST_SAMPLES = 50

    y_true = valid_targets[:NUMBER_TEST_SAMPLES]
    y_score = []
    for index in range(NUMBER_TEST_SAMPLES): #compute one at a time due to memory constraints
        probs = predict(img_path = valid_files[index])
        y_score.append(probs)
    correct = np.array(y_true) == np.array(y_score)
    acc = (np.mean(correct)*100)
    print("Accuracy of current batch: ", acc)
    # Return the negative accuracy as the fitness value
    return -acc

# Define the lower and upper bounds of the search space for learning rate and batch size
# lb = [1e-6, 8, 0.5, 0.9, 1e-7]
# ub = [1e-2, 32, 1, 1, 1e-6]
lb = [1e-6]
ub = [1e-2]

# Run the PSO algorithm to optimize the parameters
xopt, fopt = pso(fitness, lb, ub, swarmsize=2, maxiter=1)

# Print the optimal parameters and accuracy
print('Optimal learning rate:', xopt[0])
# print('Optimal batch size:', int(xopt[1]))
print('Optimal accuracy:', -fopt)